In [40]:
!pip install transformers

In [41]:
import transformers

In [42]:
text="Sometimes that's just the way it has to be. Sure, there were probably other options, but he didn't let them enter his mind. It was done and that was that. It was just the way it had to be."
print(text)

Sometimes that's just the way it has to be. Sure, there were probably other options, but he didn't let them enter his mind. It was done and that was that. It was just the way it had to be.


In [43]:
from transformers import pipeline
classifier = pipeline ("text-classification")

[transformers] No model was supplied, defaulted to distilbert/distilbert-base-uncased-finetuned-sst-2-english and revision 714eb0f.
Using a pipeline without specifying a model name and revision in production is not recommended.


Loading weights:   0%|          | 0/104 [00:00<?, ?it/s]

In [44]:
outputs = classifier(text)

In [45]:
import pandas as pd
pd.DataFrame(outputs)



,label,score
0,NEGATIVE,0.82261


In [46]:
ner_tagger = pipeline("ner", aggregation_strategy="simple")
outputs = ner_tagger(text)
pd.DataFrame(outputs)

[transformers] No model was supplied, defaulted to dbmdz/bert-large-cased-finetuned-conll03-english and revision 4c53496.
Using a pipeline without specifying a model name and revision in production is not recommended.


Loading weights:   0%|          | 0/391 [00:00<?, ?it/s]

[transformers] BertForTokenClassification LOAD REPORT from: dbmdz/bert-large-cased-finetuned-conll03-english
Key                      | Status     |  | 
-------------------------+------------+--+-
bert.pooler.dense.weight | UNEXPECTED |  | 
bert.pooler.dense.bias   | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


""


In [47]:
from transformers import AutoModelForSeq2SeqLM, AutoTokenizer

# Load a lightweight T5 model and its corresponding tokenizer
model_name = "t5-small"
model = AutoModelForSeq2SeqLM.from_pretrained(model_name)
tok = AutoTokenizer.from_pretrained(model_name)

# Prepare input text with the T5 summarization prefix
input_text = "summarize: " + text
inputs = tok(input_text, return_tensors="pt", max_length=512, truncation=True)

# Generate summary
summary_ids = model.generate(
    inputs["input_ids"],
    max_length=56,
    min_length=10,
    length_penalty=2.0,
    num_beams=4,
    early_stopping=True
)

summary = tok.decode(summary_ids[0], skip_special_tokens=True, clean_up_tokenization_spaces=True)
print("summary_text:", summary)

Loading weights:   0%|          | 0/131 [00:00<?, ?it/s]

summary_text: sometimes that's just the way it has to be. it was done and that was that.


In [50]:
import torch
from transformers import AutoModelForQuestionAnswering, AutoTokenizer

# Load a robust, lightweight QA model and tokenizer
qa_model_name = "distilbert-base-cased-distilled-squad"
qa_model = AutoModelForQuestionAnswering.from_pretrained(qa_model_name)
qa_tokenizer = AutoTokenizer.from_pretrained(qa_model_name)

# Define the question and context
question = "What keeps the doctor away?"
context = "text"

# Tokenize inputs
inputs = qa_tokenizer(question, context, return_tensors="pt")

# Get model predictions
with torch.no_grad():
    outputs = qa_model(**inputs)

# Decode the answer
answer_start = torch.argmax(outputs.start_logits)
answer_end = torch.argmax(outputs.end_logits) + 1

answer = qa_tokenizer.convert_tokens_to_string(
    qa_tokenizer.convert_ids_to_tokens(
        inputs["input_ids"][0][answer_start:answer_end]
    )
)

print(f"Question: {question}")
print(f"Answer: {answer}")

Loading weights:   0%|          | 0/102 [00:00<?, ?it/s]

Question: What keeps the doctor away?
Answer: text
